In [1]:
from datasets import load_dataset
from huggingface_hub import notebook_login
from fongbe.utils import text_tools

/home/nolimits/fongbe/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import get_dataset_config_names
from huggingface_hub import hf_hub_download

REPO_ID= "nolimitsxl/fongbe-challenge"

readme_path = hf_hub_download(
    REPO_ID,
    "README.md",
    repo_type="dataset",
    revision="main",
)
print(open(readme_path, encoding="utf-8").read())

---
license: cc-by-nc-4.0
task_categories:
  - automatic-speech-recognition
language:
  - yo
  - ig
  - ha
  - fon
  - ee
configs:
  - config_name: ffstc_fra
    data_files:
      - split: train
        path: "data/corpus=ffstc/split=train/language=fra/*.parquet"
      - split: validation
        path: "data/corpus=ffstc/split=validation/language=fra/*.parquet"

  - config_name: fon_hau_fon
    data_files:
      - split: train
        path: "data/corpus=fon_hau/split=train/language=fon/*.parquet"

  - config_name: fongbe-speech-zenodo_fon
    data_files:
      - split: test
        path: "data/corpus=fongbe-speech-zenodo/split=test/language=fon/*.parquet"
      - split: train
        path: "data/corpus=fongbe-speech-zenodo/split=train/language=fon/*.parquet"
      - split: validation
        path: "data/corpus=fongbe-speech-zenodo/split=validation/language=fon/*.parquet"

  - config_name: open-bible-speech-african_ewe
    data_files:
      - split: test
        path: "data/corpus=open-

In [2]:
notebook_login()

In [5]:
import datasets
import huggingface_hub
from huggingface_hub import DatasetCard

card = DatasetCard.load(readme_path)

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("card configs:", card.data.get("configs"))

datasets: 5.2.0
huggingface_hub: 2.4.0
card configs: [{'config_name': 'ffstc_fra', 'data_files': [{'split': 'train', 'path': 'data/corpus=ffstc/split=train/language=fra/*.parquet'}, {'split': 'validation', 'path': 'data/corpus=ffstc/split=validation/language=fra/*.parquet'}]}, {'config_name': 'fon_hau_fon', 'data_files': [{'split': 'train', 'path': 'data/corpus=fon_hau/split=train/language=fon/*.parquet'}]}, {'config_name': 'fongbe-speech-zenodo_fon', 'data_files': [{'split': 'test', 'path': 'data/corpus=fongbe-speech-zenodo/split=test/language=fon/*.parquet'}, {'split': 'train', 'path': 'data/corpus=fongbe-speech-zenodo/split=train/language=fon/*.parquet'}, {'split': 'validation', 'path': 'data/corpus=fongbe-speech-zenodo/split=validation/language=fon/*.parquet'}]}, {'config_name': 'open-bible-speech-african_ewe', 'data_files': [{'split': 'test', 'path': 'data/corpus=open-bible-speech-african/split=test/language=ewe/*.parquet'}, {'split': 'train', 'path': 'data/corpus=open-bible-speec

In [9]:
from huggingface_hub import DatasetCard, HfFileSystem
from datasets import load_dataset

repo = "nolimitsxl/fongbe-challenge"
fs = HfFileSystem()
card = DatasetCard.load(repo)

# 1. Verify every path in README.md matches real files
for cfg in card.data["configs"]:
    for df in cfg["data_files"]:
        matches = fs.glob(f"datasets/{repo}/{df['path']}")
        if not matches:
            print(f"MISSING: {cfg['config_name']} [{df['split']}] -> {df['path']}")
        else:
            print(f"OK ({len(matches)} files): {cfg['config_name']} [{df['split']}]")

# 2. Retry load_dataset bypassing local cache
ds = load_dataset(
    repo,
    "zindi_fon",
    split="train",
    streaming=True,
    download_mode="force_redownload",
)
print(next(iter(ds)))

OK (3 files): ffstc_fon [train]
OK (1 files): ffstc_fon [validation]
OK (11 files): fon_hau_fon [train]
OK (1 files): fongbe-speech-zenodo_fon [test]
OK (1 files): fongbe-speech-zenodo_fon [train]
OK (1 files): fongbe-speech-zenodo_fon [validation]
OK (2 files): open-bible-speech-african_ewe [test]
OK (32 files): open-bible-speech-african_ewe [train]
OK (2 files): waxalnlp_ewe [test]
OK (9 files): waxalnlp_ewe [train]
OK (106 files): waxalnlp_ewe [unlabeled]
OK (2 files): waxalnlp_ewe [validation]
OK (1 files): youversion-african-speech_ewe [train]
OK (5 files): zindi_fon [test]
OK (82 files): zindi_fon [train]
OK (2 files): zindi_fon [validation]
{'text': 'sɔ ɖe1 wa1 yi ɔ un yi do to e nɔ gbe1 je1 lan mɛ nu1 mɛ kpɔn1 do1 gan ji1 lo bɔ e1 mɔ tɛn lo bo kpɔn1 nu3 e ɖo a gba za ce1 mɛ nu3 mi3 e', 'audio': {'bytes': b'RIFF\xa4 \x05\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\x80 \x05\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

In [4]:
train_data = load_dataset(
    "parquet",
    data_files={
        "train": (
            f"hf://datasets/{REPO_ID}/"
            "data/corpus=zindi/split=train/language=fon/*.parquet"
        )
    },
    split="train",
    streaming=True,
)

In [1]:
from fongbe.dataloader import get_fongbe_loader

train, validation = get_fongbe_loader(None, 32, 60)

/home/nolimits/fongbe/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
for x, y in train:
    print(y)

AttributeError: 'NoneType' object has no attribute 'text_to_ids'

In [ ]:
transcripts = ["ke2mi na1 yi azɔ3mɛ ɖo canada e1 na1 yi1 ɖo semaine e ja3we ɔ1 mɛ",
"un mɔ nyavi1 ɖokpo1 bɔ e1 ka1 nyi1 nyavi1 vɔvɔ ɖe1 lobɔ e1 ɖo sin ɖe do1 nɛgbe1 wɛ",
"vi3 kpɛvi1 malɛnu vi1 lɛ1ɛ tɔn atɔn bo do1 ja1la1bu",
"lamine mɔ kwin1 syɛn1syɛn1 kpɛvi1 ɖe1 ɖo abɔ3ta tɔn",
"rodrigue ɔ1 e1 lɛ fɛnnyɛ sin1 ama bo mɔtɛn bɔ wu3klu1klu1 tɔn gbɔ",
"rose nɔ gbi1di1 ɖo1 asa1 tɔn lɛ1ɛ ji1 ami e blo3 nu1 kɛn1kun1tin1ma",
"aklu1nɔzan1gbe ɔ1 azan3 ɖe1wɛ e1 nyi1 bɔ e2 nɔ do1 nu3 yɔ3yɔ1 yi ami3sa",
"a kwɛ1 caki atɔ1ɔn1 gbe ɖi1e caki atɔ1ɔn1 gbe",
"a sya1 e1lɔ1 e1 nyɔ1 kpɔn1 ɖe1su1 titewungbe bo1 sɛ acɔ1 ganji1",
"atin1kɛnkwin1 nabi1 wɛ un ka ɖo1 na1 nu ɖo e1gbe sin1 azan3 ɔ ji1 ɖo gan atɔn mɛ hwelɛ1kɔ",
"atin1kɛn1sa1tɔ ɔ1 ɖɔ jle1 que hunmasin ɔ1 e1 nyɔ1 nu1 nyɔ3nu ɖe3 ɖo adɔgo ɔ1",
"ayikun1 lin1fin kpo1 agbade1 lin1fin kpo1 wɛ e2 nɔ sɔ1 bo nɔ sɔ1 blo1 ata1 na1",
"azɔngbo1 e1lɔ1 ɖo xasun mɛ bo1 ɖo kɔ1mɛ",
"blo3 kɛn1 e1e e2 nɔ ylɔ1 mɛ na1 ɔ1 caki afɔtɔn nu do1 alo3kan ce e1lɔ ji1 ya3ya1 ɖokpo1 kanwe tan1tɔn wo3 kanɖe1 gban atɔ1ɔn1 tɔn nukun1 atɔn kanɖe1",
"bɔ3lu afɔsɔ1xokplɔn1mɛtɔ1 ɔ1 ɖɔ mi3 ni1 nɔ1 kplɔn1 kan miɖe1e lɛ1ɛ ɖo zan3zan3 lɛ1ɛ bi3 nu1 ganxixo atɔ1ɔn1 hwɛ3 co1bo1 nɔ ɖu nu3",
"cɔ1fu e1e ɖo xa1 mi3tɔn mɛ ɔ1 e1 nɔ1 hun gan tan1tɔn mɛ lɛ1ɛ bi3",
"de3 un wa azɔ3 ɖo gle ji1 ɖo ke1ze ɔ1 bi3 mɛ kpo1 sɛ1na1mi kpo1"
]


In [ ]:
denorm_transcripts = []
for transcript in transcripts:
    denorm = text_tools.desyllabify(transcript)
    denorm_transcripts.append(denorm)

In [ ]:
with open("../data/transcript_manifest (1).txt","r") as f:
    print(len(f.readlines()))